In [1]:
import tensorflow as tf #donne des outils pour construire et  entrainer
from tensorflow.keras import layers,models
# print(tf.__version__)


# construire cnn
model = models.Sequential([
    layers.Input(shape=(224,224,3)),#img RGB de 224 x 224 pixels
    # 1ere bloc de convolution
    layers.Conv2D(32,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),#reduire la taille des caracteristiques

    # 2eme bloc de convolution
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),

    # 3eme
    layers.Conv2D(128,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
   
    #limiter le overfitting
    layers.Dropout(0.5),

    #trasformer les caratere en vecteur
    layers.Flatten(),

    #couche dense distinguer les class
    layers.Dense(128,activation="relu"),
    # sortie  => 4 class
    layers.Dense(4,activation="softmax")

])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,476 (42.61 MB)

 Trainable params: 11,169,476 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [2]:
# compiler le model
model.compile(
    optimizer = "adam",
    loss = "sparse_categorical_crossentropy",
    metrics =["accuracy"]
)

In [3]:
import numpy as np

X_train = np.load("../data/processed/X_train.npy")
X_val = np.load("../data/processed/X_val.npy")
X_test = np.load("../data/processed/X_test.npy")

y_train = np.load("../data/processed/y_train.npy")
y_val = np.load("../data/processed/y_val.npy")
y_test = np.load("../data/processed/y_test.npy")

print("X_train :", X_train.shape)
print("X_val :", X_val.shape)
print("X_test :", X_test.shape)

X_train : (4708, 224, 224, 3)
X_val : (1009, 224, 224, 3)
X_test : (1009, 224, 224, 3)


In [4]:
# entrainement
from tensorflow.keras.callbacks import EarlyStopping,ModelCheckpoint
import os

os.makedirs("../models", exist_ok=True)

# arret si la validation ne s'ameliore plus

early_stop=EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

# save le best model
checkpoint = ModelCheckpoint(
    "../models/best_brainscan.keras",
    monitor="val_loss",
    save_best_only=True
)

history= model.fit(
    X_train,
    y_train,
    validation_data=(X_val,y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stop,checkpoint]
)

Epoch 1/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 130s 850ms/step - accuracy: 0.7449 - loss: 0.6340 - val_accuracy: 0.8325 - val_loss: 0.4346
Epoch 2/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 118s 799ms/step - accuracy: 0.8672 - loss: 0.3455 - val_accuracy: 0.8751 - val_loss: 0.3550
Epoch 3/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 143s 965ms/step - accuracy: 0.9106 - loss: 0.2395 - val_accuracy: 0.8375 - val_loss: 0.4919
Epoch 4/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 145s 981ms/step - accuracy: 0.9393 - loss: 0.1565 - val_accuracy: 0.9088 - val_loss: 0.3008
Epoch 5/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 120s 807ms/step - accuracy: 0.9558 - loss: 0.1317 - val_accuracy: 0.8731 - val_loss: 0.4558
Epoch 6/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 162s 1s/step - accuracy: 0.9698 - loss: 0.0883 - val_accuracy: 0.9217 - val_loss: 0.2763
Epoch 7/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 152s 1s/step - accuracy: 0.9892 - loss: 0.0369 - val_accuracy: 0.9128 - val_loss: 0.3219
Epoch 8/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 158s 1s/step - accuracy: 0.9919 - loss: 0.

In [7]:
from tensorflow.keras.models import load_model

best_model = load_model("../models/best_brainscan.keras")

test_loss,test_accuracy=best_model.evaluate(
    X_test,y_test
)
print("test loss :",test_loss)
print("test accuracy :",test_accuracy)

32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 153ms/step - accuracy: 0.9366 - loss: 0.1863
test loss : 0.1863452047109604
test accuracy : 0.9365708827972412


In [8]:
import pickle
import os
# save historique de train
os.makedirs("../data/processed", exist_ok=True)

with open("../data/processed/history.pkl","wb") as f :
    pickle.dump(history.history,f)
    

In [9]:
# save model final

import os

os.makedirs("../models", exist_ok=True)

model.save("../models/brainscan_cnn.keras")